# Query by Committee in Active Learning - Starter Notebook
This notebook uses a committee of diverse classifiers and selects samples with highest vote disagreement.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

In [ ]:
X, y = make_classification(n_samples=1100, n_features=10, n_informative=7, random_state=42)
X_pool, _, y_pool, _ = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

X_labeled = X_pool[:120]
y_labeled = y_pool[:120]
X_unlabeled = X_pool[120:]

committee = [
    LogisticRegression(max_iter=1000),
    RandomForestClassifier(n_estimators=120, random_state=42),
    GaussianNB(),
]
for member in committee:
    member.fit(X_labeled, y_labeled)

In [ ]:
votes = np.column_stack([member.predict(X_unlabeled) for member in committee])
vote_disagreement = votes.var(axis=1)
query_indices = np.argsort(vote_disagreement)[-20:]

result = pd.DataFrame({
    'query_index': query_indices,
    'vote_pattern': [votes[i].tolist() for i in query_indices],
    'disagreement_score': vote_disagreement[query_indices],
}).sort_values('disagreement_score', ascending=False)
result.head(10)